# 02 · Préparation des données et protocole d'évaluation

Étapes 3 et 4. Le code réutilisable est dans `src/inclusion_financiere/` (`data.py`, `features.py`, `evaluate.py`).

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from inclusion_financiere.config import RANDOM_STATE, FIGURES, DECISIONS, COST_FP, COST_FN
from inclusion_financiere.data import load_raw, clean, split, make_cv, CAT_COLS, NUM_COLS

FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)

## Étape 3 · Séparer X et y, construire les variables

In [2]:
from inclusion_financiere.features import (AddFeatures, onehot_preprocessor, pipe,
                                           CAT_FE, NUM_FE, CAT_OWN, NUM_OWN)

X, y = clean(load_raw())
print("X :", X.shape, "| colonnes :", list(X.columns))
AddFeatures().fit_transform(X.head(3))[CAT_FE[-2:] + NUM_FE[2:]]

X : (23524, 10) | colonnes : ['country', 'location_type', 'cellphone_access', 'household_size', 'age_of_respondent', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type']


,age_group,household_group,education_rank,is_head,formal_job,urban_phone,is_alone
0,16-25,3-5,2,0,0,0,0
1,66+,3-5,0,1,0,0,0
2,26-35,3-5,3,0,0,1,0


## Étape 4 · Mettre le test sous scellé

In [3]:
X_train, X_test, y_train, y_test = split(X, y)
print("X_train :", X_train.shape, "| prévalence", round(y_train.mean(), 4))
print("X_test  :", X_test.shape, "| prévalence", round(y_test.mean(), 4))
assert (len(X_train), len(X_test)) == (18819, 4705)   # point de contrôle
del X_test, y_test   # plus utilisés jusqu'à l'étape 10 (notebook 06)

X_train : (18819, 10) | prévalence 0.1408
X_test  : (4705, 10) | prévalence 0.1407


Ce que le préprocesseur **apprend** sur les données d'entraînement (ici ajusté une fois sur `X_train` pour l'inspecter ; en validation croisée il est réajusté dans chaque pli, via le pipeline) :

In [4]:
prep = pipe("passthrough")[:-1].fit(X_train, y_train)
ohe = prep[-1].named_transformers_["cat"]
print("Nombre de colonnes après encodage :", len(prep[-1].get_feature_names_out()))
for col, inf in zip(CAT_FE, ohe.infrequent_categories_):
    if inf is not None:
        print(f"Modalités regroupées en « infrequent » pour {col}: {list(inf)}")

Nombre de colonnes après encodage : 53
Modalités regroupées en « infrequent » pour marital_status: ['Dont know']


**Q3.1** — `AddFeatures` **n'apprend rien** (règles fixées à la main, `fit` renvoie `self`). En revanche :
- `OneHotEncoder` apprend **la liste des catégories** et, avec `min_frequency=20`, lesquelles sont rares ;
- `StandardScaler` apprend **la moyenne et l'écart-type** de chaque variable numérique.

Si on les ajuste sur tout le jeu avant la validation croisée, le pli de validation influence ces paramètres : c'est une **fuite** qui rend le score optimiste (slide 23). Dans un pipeline, `cross_validate` réajuste chaque étape **dans chaque pli, sur le seul pli d'entraînement**, et ne fait que `transform` sur le pli de validation : on simule l'arrivée de données vraiment nouvelles. (Ci-dessus, l'ajustement sur `X_train` n'a servi qu'à inspecter le résultat.)

**Q3.2** — `min_frequency=20` : les modalités vues moins de 20 fois à l'entraînement sont regroupées dans une colonne unique « infrequent » au lieu d'avoir chacune leur colonne (ici `marital_status = "Dont know"`, 8 lignes au total) : moins de colonnes quasi vides, des coefficients moins instables. `handle_unknown="infrequent_if_exist"` : une modalité **jamais vue** à l'entraînement (fréquent pour une modalité rare dans un pli, ou en production) est envoyée dans cette catégorie « infrequent » si elle existe (sinon encodée tout à zéro), au lieu de **faire planter** le pipeline. C'est la réponse directe au problème soulevé en Q2.6.

**Q3.3** —
- **Régression logistique** : le **rang ordinal** (`education_rank`) donne un seul coefficient « chaque niveau en plus » : parcimonieux et stable, mais il impose un effet linéaire. Le **one-hot** est plus souple (un coefficient par niveau). Les deux ensemble se complètent : le rang capte la tendance, le one-hot les écarts à la tendance (régularisation L2 pour gérer la redondance).
- **Arbre** : le **rang** aide le plus : une seule coupure (« rang ≥ 3 ») isole les niveaux élevés, alors qu'avec le one-hot il faut plusieurs coupures successives sur des colonnes séparées. Un arbre n'a pas besoin de linéarité, l'ordre lui suffit.

**Q3.4** — **Forêt aléatoire : inutile.** Un arbre compare une variable à un seuil ; toute transformation croissante (comme la standardisation) laisse les coupures identiques. **Régression logistique : utile.** Le solveur converge mieux avec des échelles comparables et surtout la **pénalité L2** (par défaut) pénalise les coefficients de façon équitable seulement si les variables sont à la même échelle : sans cela, `age_of_respondent` (16 à 100) et `is_alone` (0/1) seraient régularisés très différemment. Les coefficients deviennent aussi comparables entre eux.

**Q3.5** — En plus des variables du guide (`education_rank`, `is_head`, `formal_job`, `urban_phone`, `age_group`), j'ajoute deux variables de mon cru (`CAT_OWN`, `NUM_OWN` dans `features.py`) :
- `is_alone` : la personne vit seule (`household_size == 1`). Une personne seule gère elle-même revenus et dépenses et ne peut pas s'appuyer sur le compte d'un autre membre du ménage.
- `household_group` : taille du ménage par tranches (0–2, 3–5, 6–9, 10+). Les grands ménages, plus souvent ruraux et pauvres, ont un revenu par personne plus faible ; la relation n'est pas forcément linéaire, d'où le découpage.

(L'ancienne variable `is_rural` a été supprimée : c'était une copie exacte de `location_type`, déjà encodée.) L'effet est mesuré à l'étape 5 (Q5.2).

## 4.2 · Validation croisée stratifiée, un seul objet pour tous

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from inclusion_financiere.evaluate import summarize

cv = make_cv()
scoring = {"pr_auc": "average_precision", "roc_auc": "roc_auc",
           "brier": "neg_brier_score", "logloss": "neg_log_loss", "acc": "accuracy"}
logreg = pipe(LogisticRegression(max_iter=2000))
res = cross_validate(logreg, X_train, y_train, cv=cv, scoring=scoring,
                     return_train_score=True)
summarize(res, "logreg", list(scoring)).to_frame().T.set_index("model").round(4).T

model,logreg
pr_auc_val_mean,0.55948
pr_auc_val_std,0.028403
pr_auc_train_mean,0.565384
pr_auc_train_std,0.00665
roc_auc_val_mean,0.848999
roc_auc_val_std,0.01049
roc_auc_train_mean,0.852929
roc_auc_train_std,0.002527
brier_val_mean,0.087026
brier_val_std,0.003044


## 4.3 · Généraliser à un pays jamais vu

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
groups = X_train["country"]
pays, prev = [], []
for i, (tr, va) in enumerate(sgkf.split(X_train, y_train, groups)):
    print(i, list(groups.iloc[va].unique()), round(y_train.iloc[va].mean(), 3))
    pays.append(groups.iloc[va].unique()[0]); prev.append(y_train.iloc[va].mean())

res_pays = cross_validate(logreg, X_train, y_train, groups=groups, cv=sgkf,
                          scoring=["average_precision", "roc_auc"])
par_pays = pd.DataFrame({"pays": pays, "prevalence": prev,
                         "pr_auc": res_pays["test_average_precision"],
                         "roc_auc": res_pays["test_roc_auc"]})
par_pays["pr_auc / prevalence"] = par_pays["pr_auc"] / par_pays["prevalence"]
print(par_pays.round(3).to_string(index=False))
print(f"\nCV stratifiée : PR-AUC {res['test_pr_auc'].mean():.3f} ± {res['test_pr_auc'].std(ddof=1):.3f}"
      f" | ROC-AUC {res['test_roc_auc'].mean():.3f} ± {res['test_roc_auc'].std(ddof=1):.3f}")

0 ['Rwanda'] 0.116
1 ['Tanzania'] 0.092
2 ['Uganda'] 0.089
3 ['Kenya'] 0.248


    pays  prevalence  pr_auc  roc_auc  pr_auc / prevalence
  Rwanda       0.116   0.406    0.793                3.514
Tanzania       0.092   0.420    0.842                4.581
  Uganda       0.089   0.491    0.834                5.515
   Kenya       0.248   0.625    0.816                2.517

CV stratifiée : PR-AUC 0.559 ± 0.028 | ROC-AUC 0.849 ± 0.010


**Q4.1** — La stratification impose la **même proportion de positifs** dans le train, le test et chaque pli (ici 0,1408 et 0,1407). Avec 2 % de positifs et des plis de 200 lignes, un pli contient en moyenne **4 positifs** ; sans stratification, le tirage peut en donner 0, 1 ou 8. Avec 0 positif, recall et PR-AUC ne sont pas définis ; avec 1 ou 2, ils sont extrêmement instables. Les écarts entre plis refléteraient le hasard du découpage plutôt que la qualité du modèle, et l'écart-type exploserait.

**Q4.2** — Pour que tous les modèles soient évalués sur **exactement les mêmes plis** : les différences de score viennent alors des modèles et non d'un découpage plus ou moins favorable. Cela permet surtout une **comparaison appariée, pli par pli** (slide 52) : le pli « difficile » l'est pour tous, et la différence entre deux modèles est beaucoup moins bruitée que la différence de leurs moyennes (l'écart-type entre plis, ~0,028 ici, est bien plus grand que les écarts entre bons modèles). C'est aussi une garantie de reproductibilité (`make_cv()`, graine 42, défini une seule fois dans `data.py`).

**Q4.3** — Avec 86 % de négatifs, la ROC-AUC est **flattée** : le taux de faux positifs (FP / tous les négatifs) reste faible même quand le modèle émet beaucoup de fausses alertes, car le dénominateur est énorme. La PR-AUC se concentre sur la classe positive (précision et recall), ce qui correspond à la question de KaziPay : « parmi les gens que je contacte, combien sont bancarisés, et combien en ai-je trouvé ? ». Un modèle **au hasard** a une PR-AUC égale à la **prévalence, ≈ 0,141** (contre 0,5 en ROC-AUC). La régression logistique (0,559 ± 0,028) fait donc environ **4 fois mieux que le hasard**.

**Q4.4** — La boucle confirme qu'un seul pays est laissé de côté par pli (Rwanda, Tanzanie, Ouganda, Kenya). En CV par pays, la **ROC-AUC** va de 0,793 (Rwanda) à 0,842 (Tanzanie), toujours **sous** la CV stratifiée (0,849 ± 0,010) : généraliser à un pays inconnu coûte un peu de performance. La **PR-AUC** va de 0,406 à 0,625, mais elle **n'est pas comparable** d'un pays à l'autre car sa valeur « au hasard » est la prévalence du pays, de 0,089 (Ouganda) à 0,248 (Kenya). Rapportée à la prévalence, le Kenya, qui a la PR-AUC la plus haute (0,625), a en fait le gain le plus faible (×2,5) et l'Ouganda le plus fort (×5,5).

**Pour un déploiement au Sénégal** (slide 28) : ce test est encourageant (le modèle garde un pouvoir discriminant sur un pays jamais vu) mais insuffisant. Les quatre pays sont d'Afrique de l'Est, enquêtés en 2016–2018, avec des systèmes bancaires et de mobile money différents de ceux de l'UEMOA ; la variable `country` n'aurait pas de modalité pour le Sénégal et la prévalence y est inconnue. Il faudrait collecter un échantillon local étiqueté, y mesurer la performance, **recalibrer** les probabilités et réajuster le seuil avant tout usage, et ne pas présenter les scores de la CV stratifiée comme la performance attendue.

**Q4.5** — **Non.** `TimeSeriesSplit` sert quand on prédit le futur à partir du passé sur des observations ordonnées et dépendantes dans le temps : mélanger ferait fuiter le futur dans le passé. Ici, ce sont des enquêtes en **coupe transversale** : chaque personne n'apparaît qu'une fois, il n'y a pas de série temporelle. Surtout, l'année est **confondue avec le pays** (Q2.3) : un découpage par année reviendrait à un découpage par pays, déjà traité en 4.3, et entraînerait sur le Rwanda pour prédire le Kenya. `year` a d'ailleurs été retirée. Un découpage temporel deviendrait pertinent si l'on disposait de plusieurs vagues d'enquête **dans le même pays** et que l'on voulait prédire la vague suivante.